# Exercise II: Regression - portable notebook

This is the **portable version** of the Exercise II regression practice
from **Machine Learning for Neuroscience**, generated from the canonical
course notebook by `scripts/build_portable_notebook.py`. It is meant for
running or editing the code in Google Colab or in a local VS Code /
Jupyter setup.

The richer version -- with the feature-set comparison activity embedded
and running in the browser -- is the published course page:
<https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_02/exercise_02.html>

In this notebook the interactive activity is replaced by a link to that
page; every Python analysis cell is kept and runnable. Questions marked
*Think first* are followed, where one exists, by a collapsible *Check
your reasoning* block; open questions are left without one fixed answer.

## Setup

This notebook imports only `numpy`, `pandas`, `matplotlib` and
`scikit-learn`. All four are already installed on Google Colab, and in a
typical scientific-Python environment, so there is normally nothing to
do here.

If one of the imports further down fails, run the next cell once (edit
the version pins if your project needs specific ones), then restart the
kernel and run the notebook from the top. The notebook also downloads two
public data files the first time it runs, so it needs internet access.

In [ ]:
# If an import below fails, uncomment and run this line once, then
# restart the kernel. Safe on Colab, VS Code and Jupyter.
# %pip install numpy pandas matplotlib scikit-learn

# Exercise II: Regression

## What this notebook covers

This is the linear-regression half of Exercise II. It follows the lecture, so it
is short on theory and long on practice. You will:

1. load a wide *modelling table* -- ABIDE-II phenotype columns joined to
   FreeSurfer brain measurements for the same participants;
2. build one honest linear-regression workflow: split, fit on training data,
   predict held-out data, score it;
3. compare that honest score with the *training* score and with a deliberately
   *invalid* fit-on-the-test-set score, to see what "evaluation" can and cannot
   mean;
4. use a browser activity to compare feature sets side by side;
5. see what changes -- and what does not -- when the sample gets smaller.

**Prerequisites:** the regression lecture; comfort with `pandas`, `numpy`,
`matplotlib`, and the scikit-learn `fit` / `predict` pattern. Other regression
models come in the next practice.

## 1. The modelling table

Each **row is one participant**. The columns fall into three kinds:

- **phenotype columns** -- possible outcomes and context: `age`, `sex`,
  diagnostic `group`, and cognitive / behavioural scores such as `FIQ`
  (full-scale IQ) merged in from the ABIDE-II phenotypic file;
- **brain columns** -- one FreeSurfer measurement (`fsCT` cortical thickness,
  `fsArea` surface area, `fsVol` grey-matter volume, `fsLGI` gyrification) for
  one cortical region of one hemisphere, e.g. `fsCT_L_46_ROI`. The regions are
  the 360 parcels of the HCP-MMP1 (Glasser) atlas;
- **identifiers** -- `subject`, `site`.

This is a *modelling* table: wide, one measurement per column, ready for
`X` / `y`. It is not the small phenotype-only table from Exercise I.

In [ ]:
# Data loading. In the published book this cell is collapsed; it is plain,
# runnable Python -- two public CSVs pinned to an immutable commit, merged on the
# participant id. Nothing here is repository-specific.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error

PIN = "e4eed3c4daa7f40b0ba931182a8c7e5e691dba6b"
BASE = f"https://raw.githubusercontent.com/neurohackademy/nh2020-curriculum/{PIN}/tu-machine-learning-yarkoni/data"

brain = pd.read_csv(f"{BASE}/abide2.tsv", sep="\t")
brain = brain.loc[:, [c for c in brain.columns if not str(c).startswith("Unnamed")]]
phen = pd.read_csv(f"{BASE}/abide2_phenotypic.csv", encoding="latin-1", low_memory=False)
phen.columns = phen.columns.str.strip()

KEEP_PHEN = ["FIQ", "VIQ", "PIQ", "SRS_TOTAL_RAW", "ADOS_G_TOTAL", "ADI_R_SOCIAL_TOTAL_A"]
model_df = brain.merge(phen[["SUB_ID", *KEEP_PHEN]], left_on="subject", right_on="SUB_ID", how="left")
model_df = model_df.drop(columns=["SUB_ID"])

BRAIN_COLS = [c for c in model_df.columns if c.startswith("fs")]
PHENO_COLS = ["age", "age_resid", "sex", "group", *KEEP_PHEN]
print(f"modelling table: {model_df.shape[0]} participants x {model_df.shape[1]} columns")

In [ ]:
# A compact look: the phenotype columns plus three representative brain columns,
# then a separate count summary. (Printing all 1446 columns would tell you
# nothing.)
def measure_of(col):
    return col.split("_")[0]          # fsCT, fsArea, fsVol, fsLGI

def roi_of(col):
    return col.split("_", 2)[2].rsplit("_ROI", 1)[0]   # "L_46" etc.

preview_cols = ["subject", "site", "group", "age", "sex", "FIQ",
                "fsCT_L_46_ROI", "fsArea_L_IPS1_ROI", "fsVol_R_V1_ROI"]
display(model_df[preview_cols].head(4))

measures = sorted({measure_of(c) for c in BRAIN_COLS})
rois = sorted({roi_of(c) for c in BRAIN_COLS})
print(f"phenotype columns : {len(PHENO_COLS)}  -> {PHENO_COLS}")
print(f"brain features    : {len(BRAIN_COLS)}  = {len(measures)} measures x {len(rois)} region-hemispheres")
print(f"measurement types : {measures}")
print(f"missing brain cells: {int(model_df[BRAIN_COLS].isna().sum().sum())}")
print()
print("FIQ available for", int(model_df['FIQ'].notna().sum()), "of", len(model_df), "participants")

#### Think first

Before going on, name -- for this table -- (a) one column you could use as an
outcome, (b) roughly how many columns could be *features*, (c) what a single row
is, (d) the four measurement types, and (e) how a brain column encodes both a
region and a hemisphere.

<details>
<summary><strong>Check your reasoning</strong></summary>

Outcome: `FIQ` (or `age`, `SRS_TOTAL_RAW`, ...). Features: the ~1440 `fs...`
brain columns. A row: one participant. Measurement types: `fsCT`, `fsArea`,
`fsVol`, `fsLGI`. A brain column name is `fs<measure>_<hemisphere>_<region>_ROI`,
so `fsCT_L_46_ROI` is cortical thickness of left area 46.

</details>

## 2. One honest linear-regression workflow

**Target:** `FIQ`, full-scale IQ. **Features:** a small, *pre-declared* brain
set -- bilateral **cortical thickness** in a **frontoparietal** bundle of
regions. That bundle is chosen *a priori* from the intelligence literature (the
Parieto-Frontal Integration Theory; see Section 4), **not** because it scores
well here. It has 78 features -- enough to be genuinely multivariate, small
enough that ordinary least squares is well posed for ~680 training rows.

The workflow, step by step:

In [ ]:
# The pre-declared frontoparietal cortical-thickness recipe (39 Glasser regions,
# bilateral). Region list is fixed here and checked against
# book/config/abide_modeling.json by the tests.
FRONTOPARIETAL = ["46", "9-46d", "a9-46v", "p9-46v", "9a", "9p", "8C", "8Av", "8Ad", "IFJa", "IFJp", "IFSa", "IFSp", "p47r", "a47r", "8BL", "PGs", "PGi", "PFm", "PF", "PFt", "IP0", "IP1", "IP2", "IPS1", "LIPv", "LIPd", "MIP", "AIP", "7PC", "7PL", "7Pm", "7AL", "7Am", "a32pr", "p32pr", "d32", "8BM", "SCEF"]

def bundle_columns(rois, measures):
    cols = []
    for roi in rois:
        for m in measures:
            for hemi in ("L", "R"):
                name = f"{m}_{hemi}_{roi}_ROI"
                if name not in model_df.columns:
                    raise KeyError(f"expected brain column missing: {name}")
                cols.append(name)
    return cols

FEATURES = bundle_columns(FRONTOPARIETAL, ["fsCT"])
assert all(c.startswith("fsCT_") for c in FEATURES)           # brain-only
assert "FIQ" not in FEATURES and "VIQ" not in FEATURES        # no target / IQ leakage
print(f"{len(FEATURES)} features, e.g. {FEATURES[:3]}")

In [ ]:
# 1-2. brain-only X and target y; drop rows with no FIQ.
has_fiq = model_df["FIQ"].notna()
X = model_df.loc[has_fiq, FEATURES].to_numpy(float)
y = model_df.loc[has_fiq, "FIQ"].to_numpy(float)
groups = model_df.loc[has_fiq, "group"].to_numpy()      # 1 = autism, 2 = control

# 3. one fixed, reproducible split. Stratify by diagnosis so train and test have
#    a similar autism / control mix -- without turning this into a splitting
#    lecture.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=groups
)
print(f"n_train = {len(y_train)}   n_test = {len(y_test)}   n_features = {X.shape[1]}")

In [ ]:
# 4-7. scale + fit on the TRAINING rows only (Pipeline keeps the scaler honest);
#      predict the untouched test rows; score.
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
test_r2 = r2_score(y_test, y_pred)
test_mse = mean_squared_error(y_test, y_pred)
print(f"held-out R^2 = {test_r2:.3f}")
print(f"held-out MSE = {test_mse:.1f}  (RMSE {test_mse**0.5:.1f} IQ points)")

In [ ]:
# 8. observed vs predicted on the test set, with the perfect-prediction diagonal.
fig, ax = plt.subplots(figsize=(4.4, 4.4))
lims = [min(y_test.min(), y_pred.min()) - 3, max(y_test.max(), y_pred.max()) + 3]
ax.plot(lims, lims, "--", color="0.4", lw=1, label="perfect prediction")
ax.scatter(y_test, y_pred, s=16, alpha=0.5)
ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect("equal")
ax.set_xlabel("observed FIQ"); ax.set_ylabel("predicted FIQ (held-out)")
ax.set_title(f"held-out R$^2$ = {test_r2:.3f}")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

A few things to hold onto:

- the fitted model is a hyperplane in 78-dimensional feature space -- there is
  nothing useful to *draw* there. The observed-vs-predicted plot is the honest
  picture: if the model predicted well, points would hug the diagonal;
- a **negative held-out R²** is a valid result. It means the model does *worse*
  on new participants than simply predicting the training-set mean IQ for
  everyone. It is not a bug;
- this split estimates generalisation to **new participants from the same 17
  ABIDE-II sites**. A random participant split says nothing about how the model
  would do on an entirely new scanner or site.

#### Think first

The test set here is a random 25% of participants, drawn from the *same* 17
sites, scanners, and protocols as the training set. Name one prediction task
this held-out score does **not** speak to.

<details>
<summary><strong>Check your reasoning</strong></summary>

It does not estimate performance on a **new site or scanner** the model has
never seen. Sites differ in scanner hardware, sequences, and the people they
recruit; a model tuned on these 17 can lean on site-linked quirks that will not
transfer. A leave-one-site-out split would be needed to speak to that.

</details>

## 3. Three ways to score the same model

Same fixed split, same 78-feature recipe. Three numbers:

| | fit on | evaluate on | what it estimates |
|---|---|---|---|
| **A. Correct** | training rows | untouched test rows | performance on new participants |
| **B. Training score** | training rows | those same training rows | how well it fits data it has seen (a diagnostic, not a generalisation estimate) |
| **C. Invalid** | **test rows** | those same test rows | nothing usable -- the data used to fit cannot also give an honest score |

#### Think first

Predict the order of the three R² values from largest to smallest, and say why.

In [ ]:
# A: correct (already computed above) -> reuse `model`, `y_test`, `y_pred`.
train_r2 = r2_score(y_train, model.predict(X_train))          # B: resubstitution
train_mse = mean_squared_error(y_train, model.predict(X_train))

# C: a DELIBERATELY INVALID model -- fit on the test rows, scored on the same
# test rows. Named so it cannot be reused by accident.
invalid_test_fitted_model = make_pipeline(StandardScaler(), LinearRegression())
invalid_test_fitted_model.fit(X_test, y_test)
invalid_pred = invalid_test_fitted_model.predict(X_test)
invalid_r2 = r2_score(y_test, invalid_pred)
invalid_mse = mean_squared_error(y_test, invalid_pred)

scores = pd.DataFrame(
    {
        "fit on": ["training rows", "training rows", "test rows (INVALID)"],
        "evaluated on": ["test rows", "training rows", "same test rows"],
        "R^2": [test_r2, train_r2, invalid_r2],
        "MSE": [test_mse, train_mse, invalid_mse],
    },
    index=["A. correct", "B. training score", "C. invalid"],
)
print(f"n_train = {len(y_train)}   n_test = {len(y_test)}   n_features = {X.shape[1]}")
scores.round(3)

In [ ]:
panels = [
    ("A. correct\n(fit train, test test)", y_test, y_pred, test_r2),
    ("B. training score\n(fit train, score train)", y_train, model.predict(X_train), train_r2),
    ("C. invalid\n(fit test, score test)", y_test, invalid_pred, invalid_r2),
]
allv = np.concatenate([y_train, y_test, y_pred, model.predict(X_train), invalid_pred])
lims = [allv.min() - 3, allv.max() + 3]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.9), sharex=True, sharey=True)
for ax, (title, obs, pred, r2) in zip(axes, panels):
    ax.plot(lims, lims, "--", color="0.4", lw=1)
    ax.scatter(obs, pred, s=10, alpha=0.4)
    ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect("equal")
    ax.set_title(f"{title}\nR$^2$ = {r2:.3f}", fontsize=9)
    ax.set_xlabel("observed FIQ")
axes[0].set_ylabel("predicted FIQ")
plt.tight_layout(); plt.show()

- **B is not a competing model.** A high training R² next to a low test R² is
  the signature of a model fitting noise it cannot reproduce on new data.
- **C is not a competing model either.** It looks good only because the same 227
  rows were used to choose the coefficients and to grade them.
- B compares *different rows* (train vs test), so part of the gap could be a
  lucky or unlucky split. **A vs C uses the exact same test rows**, so it
  isolates the contamination directly: the only difference is whether those rows
  were also used for fitting.
- Here even the correct held-out R² is negative: with these features, in this
  sample, cortical thickness barely predicts IQ at all. Sections 4 and 5 push on
  why.

<details>
<summary><strong>Check your reasoning: the ordering</strong></summary>

C (invalid) > B (training) > A (correct). C is highest because the same 227 rows
picked the coefficients *and* graded them -- with 78 free parameters the fit can
chase noise specific to those rows. B is next: the model saw these 681 rows
while fitting, so it reproduces them better than genuinely new rows. A is lowest
because the test rows had no influence on the coefficients at all -- and with no
real signal to capture, that honest number lands below zero.

</details>

## 4. Comparing feature sets

Which brain features should predict IQ? The **Parieto-Frontal Integration
Theory** (Jung & Haier, 2007,
[doi:10.1017/S0140525X07001185](https://doi.org/10.1017/S0140525X07001185)) and
work on cortical thickness and IQ (Narr et al., 2007,
[doi:10.1093/cercor/bhl125](https://doi.org/10.1093/cercor/bhl125)) point to
dorsolateral prefrontal cortex, the inferior / superior parietal lobule and
intraparietal sulcus, and dorsal anterior cingulate. The **frontoparietal**
bundle below maps those onto the Glasser atlas labels. That is a *hypothesis
about where to look first* -- not a guarantee that those features predict IQ
here, and not proof that a better-scoring bundle is biologically causal.

The activity compares two models at a time on **one fixed cohort and one fixed
set of folds**, so any difference is a real difference between feature sets. It
shows out-of-sample scores only -- never training scores.

### Compare feature sets on the course website

The interactive activity lets you configure two linear-regression models
-- a measurement type and an anatomical ROI bundle each -- and compares
their cross-validated performance on one fixed cohort and one fixed set
of folds.

> **Interactive version on the course website.** It is embedded in the
> published Exercise II page:
> <https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_02/exercise_02.html>
> This portable notebook links to it instead of embedding it. The
> Python sections below still run the same kind of comparison directly.

#### Think first

1. Hold the ROI bundle fixed and change only the measurement type; then hold the
   measurement fixed and change only the bundle. Which control moves R² more?
2. Compare `Frontoparietal (P-FIT)` with a similarly sized comparison bundle,
   same measurement. Does the literature-motivated bundle win here?
3. Set one model to `All eligible ROIs`. What happens to the feature count and
   to R², and why?

Trying many configurations is **exploratory model comparison**. If you pick the
best-looking one and quote its cross-validated R² as "the" performance, that
number is optimistic -- you chose it *because* it looked good on this data. A
locked test set or a nested procedure is needed for an honest final claim.

## 5. What does sample size change?

Two different questions, kept separate.

### A. Two outcomes with different availability

`FIQ` and `SRS_TOTAL_RAW` (Social Responsiveness Scale total) are recorded for
different numbers of participants. Fit the **same** frontoparietal
cortical-thickness recipe and the same held-out evaluation for each.

#### Think first

If one score is predicted better than the other, is that because it is
*intrinsically* easier to predict from brain structure -- or could it be sample
size, measurement reliability, the range of scores, diagnosis mix, site mix, or
who was even given the assessment?

In [ ]:
# usable participant counts for the two targets, overall and by diagnosis group
avail = pd.DataFrame({
    tgt: {
        "usable N": int(model_df[tgt].notna().sum()),
        "autism (group 1)": int(model_df.loc[model_df["group"] == 1, tgt].notna().sum()),
        "control (group 2)": int(model_df.loc[model_df["group"] == 2, tgt].notna().sum()),
        "sites with any": int((model_df.groupby("site")[tgt].count() > 0).sum()),
    }
    for tgt in ["FIQ", "SRS_TOTAL_RAW"]
}).T
avail

In [ ]:
def holdout_r2(target, features=FEATURES, seed=42):
    m = model_df[target].notna()
    Xt = model_df.loc[m, features].to_numpy(float)
    yt = model_df.loc[m, target].to_numpy(float)
    g = model_df.loc[m, "group"].to_numpy()
    Xtr, Xte, ytr, yte = train_test_split(Xt, yt, test_size=0.25, random_state=seed, stratify=g)
    pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    return len(yt), r2_score(yte, pred), mean_squared_error(yte, pred), yt.std()

rows = []
for tgt in ["FIQ", "SRS_TOTAL_RAW"]:
    n, r2, mse, sd = holdout_r2(tgt)
    rows.append({"target": tgt, "usable N": n, "held-out R^2": round(r2, 3),
                 "held-out MSE": round(mse, 1), "score SD": round(sd, 1)})
pd.DataFrame(rows)

Comparing two *different outcomes* cannot isolate sample size: they differ in
scale, reliability, and who was assessed. Note the MSE columns are on different
scales (IQ points vs SRS raw points) and must **not** be compared directly --
that is why R² is the cross-target metric here.

### B. Match the high-N target to the low-N target

Keep `FIQ`. Repeatedly draw a training subsample the size of the usable
`SRS_TOTAL_RAW` training set, refit, and score on **one fixed FIQ held-out
set**. Many deterministic repetitions, not one lucky draw.

In [ ]:
rng = np.random.default_rng(0)

# fixed FIQ held-out set; the rest is the FIQ training pool
Xtr_pool, X_hold, ytr_pool, y_hold = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=groups
)
n_srs_train = int(round(model_df["SRS_TOTAL_RAW"].notna().sum() * 0.75))
matched_n = min(n_srs_train, len(ytr_pool))

matched_r2 = []
for _ in range(300):
    idx = rng.choice(len(ytr_pool), size=matched_n, replace=False)
    pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr_pool[idx], ytr_pool[idx])
    matched_r2.append(r2_score(y_hold, pipe.predict(X_hold)))
matched_r2 = np.array(matched_r2)

full_r2 = r2_score(y_hold, make_pipeline(StandardScaler(), LinearRegression())
                   .fit(Xtr_pool, ytr_pool).predict(X_hold))
_, srs_r2, _, _ = holdout_r2("SRS_TOTAL_RAW")
print(f"matched training N = {matched_n}   (FIQ full training N = {len(ytr_pool)})")
print(f"held-out R^2 at matched N: mean {matched_r2.mean():.3f}, "
      f"5th-95th pct [{np.percentile(matched_r2,5):.3f}, {np.percentile(matched_r2,95):.3f}]")
print(f"FIQ at full N: {full_r2:.3f}    SRS at its own N: {srs_r2:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(matched_r2, bins=30, alpha=0.8)
ax.axvline(matched_r2.mean(), color="k", lw=1, label=f"matched-N mean {matched_r2.mean():.3f}")
ax.axvline(full_r2, color="C1", lw=2, label=f"FIQ full N {full_r2:.3f}")
ax.axvline(srs_r2, color="C3", lw=2, ls="--", label=f"SRS its own N {srs_r2:.3f}")
ax.set_xlabel("held-out R$^2$"); ax.set_ylabel("repetitions")
ax.set_title(f"FIQ, training subsampled to N = {matched_n} (300 draws)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### C. Learning curve, same target

The cleanest test: keep `FIQ`, keep the recipe, keep one fixed held-out set,
and increase the training size. Resample many times at each size.

In [ ]:
sizes = [120, 200, 320, 460, 600, len(ytr_pool)]
n_rep = 40
lc_mean, lc_lo, lc_hi = [], [], []
for n in sizes:
    reps = []
    for _ in range(n_rep):
        idx = rng.choice(len(ytr_pool), size=min(n, len(ytr_pool)), replace=False)
        pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr_pool[idx], ytr_pool[idx])
        reps.append(r2_score(y_hold, pipe.predict(X_hold)))
    reps = np.array(reps)
    lc_mean.append(reps.mean()); lc_lo.append(np.percentile(reps, 10)); lc_hi.append(np.percentile(reps, 90))

print(f"n_features (p) = {X.shape[1]};  smallest training size {sizes[0]}  -> n/p = {sizes[0]/X.shape[1]:.1f}")
for n, m, lo, hi in zip(sizes, lc_mean, lc_lo, lc_hi):
    print(f"  n_train = {n:4d}   held-out R^2  mean {m:+.3f}   10-90 pct [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.fill_between(sizes, lc_lo, lc_hi, alpha=0.25, label="10th-90th percentile")
ax.plot(sizes, lc_mean, "o-", label="mean held-out R$^2$")
ax.axhline(0, color="0.5", lw=1, ls=":")
ax.set_xlabel("training-set size"); ax.set_ylabel("held-out R$^2$")
ax.set_title("FIQ learning curve (frontoparietal cortical thickness, p = 78)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

#### What the learning curve shows -- and does not

As the training set grows, the spread of held-out R² collapses: the estimate
becomes **stable**. Here the mean rises steeply out of the small-sample regime
and then flattens -- just below zero. More data made the number *trustworthy*; it
did not manufacture a signal that the features do not carry. Individual random
draws still wobble; the curve is about the average and its uncertainty, not a
promise that every draw improves.

## In summary

- An honest performance estimate fits on training data and scores on data the
  model has never seen. Scoring on the fitting rows -- whether they are the
  training set (optimistic) or, worse, the test set (invalid) -- inflates the
  number.
- Held-out R² can be negative; that is a real answer, not an error.
- More features are not automatically better: the frontoparietal bundle, and
  the "all eligible ROIs" set, do worse here than smaller comparison bundles.
- A literature-motivated feature set is a *hypothesis*. In this sample cortical
  thickness carries almost no information about IQ, wherever you look.
- Comparing two different outcomes cannot isolate sample size. Matching the
  training size and drawing a learning curve on **one** target can: they show
  that more data mainly buys *certainty*.

Next practice: k-nearest neighbours, and the bias-variance trade-off.

### Questions to take away

1. For this table, which columns are the outcome and which are the features?
2. Why does fitting and scoring a model on the *same* observations inflate its
   apparent performance? Why is fit-on-test/score-on-test worse than
   fit-on-train/score-on-train?
3. What does a *negative* held-out R² tell you?
4. Two feature sets give different cross-validated R². List three reasons other
   than "one set is biologically better" that could explain the gap.
5. Model B in Section 4 has many more features than Model A but a lower R². How
   is that possible?
6. Section 5A finds FIQ and SRS are predicted about equally poorly. Why can't
   that comparison tell you which score is "intrinsically" harder to predict?
7. In the learning curve, what improves as the training set grows, and what does
   not?